# PumpRisk Score - Data Quality Audit, Feature Engineering & Preliminary Scoring

Workflow:
1. Load 180 days of historical data for 481 micro- and small-cap stocks
2. Data quality audit – exclude inactive stocks or those with insufficient history
3. Feature engineering – calculate Price anomaly & Volume anomaly (2 of the 4 PumpRisk Score components)
4. Preliminary score leaderboard (Stage 1)
5. Determine shortlist for Stage 2 (broker/foreign flow/news/filings) based on remaining credit

In [1]:
import os
import numpy as np
import pandas as pd

CACHE_DIR = "cache"
COMBINED_PATH = os.path.join(CACHE_DIR, "daily_all_combined.csv")

ROLLING_WINDOW = 20
MIN_TRADING_DAYS = 30
MAX_ZERO_VOLUME_PCT = 0.5

df = pd.read_csv(COMBINED_PATH, parse_dates=["date"])
df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

print(f"Total rows: {len(df):,}")
print(f"Total shares: {df['symbol'].nunique()}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

Total rows: 54,150
Total shares: 481
Date range: 2026-03-16 to 2026-09-10


,symbol,date,close,open,high,low,volume,market_cap
0,ABBA,2026-03-16,39,NaN,42,37,985200,153499821423
1,ABBA,2026-03-17,41,NaN,41,39,337000,161371607137
2,ABBA,2026-03-25,43,NaN,43,41,443700,169243392851
3,ABBA,2026-03-26,44,NaN,44,41,151500,173179285708
4,ABBA,2026-03-27,40,NaN,44,40,188900,157435714280


## Data Quality Audit

Following the same pattern as Cell 11 – Stage 1 in the official Sectors (WIKA/WSKT) recipe:
stocks with an excessively short history, near-constant zero volume, or a constant price throughout the period (indicating a trading suspension or inactivity) must be excluded **before** feature engineering. Otherwise, their anomaly scores will be spurious or identical to one another (as seen in the ADCP case we encountered).

In [2]:
def audit_stock(group: pd.DataFrame) -> pd.Series:
    n_days = len(group)
    n_zero_volume = int((group["volume"] == 0).sum())
    pct_zero_volume = n_zero_volume / n_days if n_days > 0 else 1.0
    price_std = group["close"].std()
    is_constant_price = bool(pd.isna(price_std) or price_std == 0)
    return pd.Series({
        "n_days": n_days,
        "pct_zero_volume_days": pct_zero_volume,
        "is_constant_price": is_constant_price,
    })

audit_df = df.groupby("symbol").apply(audit_stock).reset_index()

excluded_mask = (
    (audit_df["n_days"] < MIN_TRADING_DAYS)
    | (audit_df["pct_zero_volume_days"] > MAX_ZERO_VOLUME_PCT)
    | (audit_df["is_constant_price"])
)
excluded = audit_df[excluded_mask].copy()
active_symbols = audit_df.loc[~excluded_mask, "symbol"].tolist()

print(f"Total initial stocks : {audit_df['symbol'].nunique()}")
print(f"Excluded (bad data) : {len(excluded)}")
print(f"Active stocks for scoring : {len(active_symbols)}")

excluded.to_csv(os.path.join(CACHE_DIR, "excluded_stocks_audit.csv"), index=False)
excluded.sort_values("pct_zero_volume_days", ascending=False).head(10)

Total initial stocks : 481
Excluded (bad data) : 74
Active stocks for scoring : 407


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_15136\564449034.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  audit_df = df.groupby("symbol").apply(audit_stock).reset_index()


,symbol,n_days,pct_zero_volume_days,is_constant_price
12,ALTO,114,1.0,True
11,ALMI,114,1.0,True
54,BEBS,114,1.0,True
57,BIKA,114,1.0,True
23,ARMY,114,1.0,True
25,ARTI,114,1.0,True
58,BIMA,114,1.0,True
66,BOSS,114,1.0,True
93,COWL,114,1.0,True
80,CBMF,114,1.0,True


In [3]:
df_active = df[df["symbol"].isin(active_symbols)].copy()
print(f"Rows after filtering: {len(df_active):,} ({df_active['symbol'].nunique()} stocks)")

Rows after filtering: 45,820 (407 stocks)


## Feature Engineering – Price & Volume Anomaly

The core principle of the PumpRisk Score is that each stock is compared **against its own historical performance** (using a rolling window) rather than against other stocks. This differs from graph-correlation approaches, which are suitable for highly correlated blue-chip stocks (such as those in the LQ45 or IDX30 indices) but less appropriate for idiosyncratic small-cap stocks.

In [4]:
def compute_features(group: pd.DataFrame, window: int = ROLLING_WINDOW) -> pd.DataFrame:
    g = group.sort_values("date").copy()
    g["return"] = g["close"].pct_change()

    roll_mean = g["return"].rolling(window, min_periods=10).mean()
    roll_std = g["return"].rolling(window, min_periods=10).std()
    g["return_zscore"] = (g["return"] - roll_mean) / (roll_std + 1e-8)

    g["volume_percentile"] = g["volume"].rolling(window, min_periods=10).apply(
        lambda x: x.rank(pct=True).iloc[-1] if len(x) > 0 else np.nan, raw=False
    )

    sma = g["close"].rolling(window, min_periods=10).mean()
    g["price_vs_sma"] = (g["close"] / sma) - 1

    recent_vol = g["return"].rolling(window, min_periods=10).std()
    hist_vol = g["return"].expanding(min_periods=10).std()
    g["vol_regime_ratio"] = recent_vol / (hist_vol + 1e-8)

    return g

df_feat = df_active.groupby("symbol", group_keys=False).apply(compute_features)
df_feat.to_csv(os.path.join(CACHE_DIR, "daily_with_features.csv"), index=False)
df_feat[["symbol", "date", "close", "volume", "return_zscore",
         "volume_percentile", "price_vs_sma", "vol_regime_ratio"]].tail(10)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_15136\3517834762.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_feat = df_active.groupby("symbol", group_keys=False).apply(compute_features)


,symbol,date,close,volume,return_zscore,volume_percentile,price_vs_sma,vol_regime_ratio
54140,ZYRX,2026-08-28,132,241900,1.014182,0.15,0.037736,0.893665
54141,ZYRX,2026-08-31,130,420600,-0.948485,0.35,0.018809,0.872268
54142,ZYRX,2026-09-01,130,307400,-0.194901,0.30,0.015228,0.868807
54143,ZYRX,2026-09-02,130,937700,-0.244803,0.70,0.010886,0.847301
54144,ZYRX,2026-09-03,130,655900,-0.202211,0.45,0.007361,0.842214
54145,ZYRX,2026-09-04,133,2728300,1.006383,0.90,0.026631,0.854888
54146,ZYRX,2026-09-07,129,1601800,-1.587145,0.85,-0.006163,0.917131
54147,ZYRX,2026-09-08,131,1221800,0.658274,0.80,0.007305,0.918097
54148,ZYRX,2026-09-09,129,167600,-0.788480,0.10,-0.008836,0.934581
54149,ZYRX,2026-09-10,129,243300,-0.087484,0.25,-0.010357,0.921675


## Latest Per-Stock Snapshot + Stage 1 Score

Take the most recent available data row for each stock (not necessarily the same absolute date for all, as some stocks may not trade on the exact same day), then combine the Price anomaly and Volume anomaly into an initial score of 0–1.

In [5]:
snapshot = df_feat.sort_values("date").groupby("symbol").tail(1).copy()

def minmax_norm(s: pd.Series) -> pd.Series:
    valid = s.dropna()
    if len(valid) < 2 or valid.max() == valid.min():
        return pd.Series(0.5, index=s.index)
    return (s - valid.min()) / (valid.max() - valid.min())

snapshot["price_anomaly_signal"] = minmax_norm(snapshot["return_zscore"].abs())
snapshot["volume_anomaly_signal"] = snapshot["volume_percentile"].fillna(0.5)

snapshot["stage1_score"] = (
    0.5 * snapshot["price_anomaly_signal"] + 0.5 * snapshot["volume_anomaly_signal"]
)

leaderboard = snapshot.sort_values("stage1_score", ascending=False).reset_index(drop=True)
leaderboard.to_csv(os.path.join(CACHE_DIR, "stage1_leaderboard.csv"), index=False)

print(f"Leaderboard Stage 1: {len(leaderboard)} stocks")
leaderboard[["symbol", "date", "close", "stage1_score",
             "price_anomaly_signal", "volume_anomaly_signal",
             "return_zscore", "volume_percentile"]].head(20)

Leaderboard Stage 1: 407 stocks


,symbol,date,close,stage1_score,price_anomaly_signal,volume_anomaly_signal,return_zscore,volume_percentile
0,JECC,2026-09-10,825,1.000000,1.000000,1.00,4.113743,1.00
1,MGNA,2026-09-10,136,0.996657,0.993313,1.00,4.086236,1.00
2,MSKY,2026-09-10,82,0.950188,0.900377,1.00,3.703918,1.00
3,VINS,2026-09-10,152,0.857666,0.715333,1.00,2.942694,1.00
4,KLIN,2026-09-10,127,0.821025,0.692049,0.95,2.846912,0.95
5,IKBI,2026-09-10,565,0.803298,0.606595,1.00,2.495377,1.00
6,SAPX,2026-09-10,406,0.772333,0.544667,1.00,2.240618,1.00
7,PLAN,2026-09-10,72,0.771339,0.592679,0.95,-2.438129,0.95
8,IDEA,2026-09-10,71,0.769195,0.538389,1.00,2.214795,1.00
9,SEMA,2026-09-10,132,0.766880,0.533760,1.00,2.195751,1.00


## Determine Stage 2 Shortlist (Budget-Aware)

Stage 2 (Broker Activity Top = 2 credits, Foreign Flow = 1 credit, News = 1 credit, Filings = 1 credit → total ~5 credits per stock) is executed only for stocks with the highest `stage1_score`, mirroring the confirmation pattern in Cells 11–12 of the official Sectors Stage 1 recipe. The shortlist size is automatically calculated based on remaining credits, incorporating a safety buffer for backtesting and re-experimentation.

In [7]:
REMAINING_CREDITS = 466
BUFFER_CREDITS = 150
CREDITS_PER_STOCK_STAGE2 = 5

usable_credits = REMAINING_CREDITS - BUFFER_CREDITS
max_shortlist_size = max(usable_credits // CREDITS_PER_STOCK_STAGE2, 0)
shortlist_size = min(max_shortlist_size, len(leaderboard))

shortlist = leaderboard.head(shortlist_size)[["symbol", "stage1_score"]].copy()
shortlist.to_csv(os.path.join(CACHE_DIR, "stage2_shortlist.csv"), index=False)

print(f"Remaining credits : {REMAINING_CREDITS}")
print(f"Buffer maintained : {BUFFER_CREDITS}")
print(f"Usable credits : {usable_credits}")
print(f"Maximum shortlist size : {max_shortlist_size} stocks")
print(f"Final Shortlist : {len(shortlist)} stocks -> cache/stage2_shortlist.csv")
shortlist.head(10)

Remaining credits : 466
Buffer maintained : 150
Usable credits : 316
Maximum shortlist size : 63 stocks
Final Shortlist : 63 stocks -> cache/stage2_shortlist.csv


,symbol,stage1_score
0,JECC,1.000000
1,MGNA,0.996657
2,MSKY,0.950188
3,VINS,0.857666
4,KLIN,0.821025
5,IKBI,0.803298
6,SAPX,0.772333
7,PLAN,0.771339
8,IDEA,0.769195
9,SEMA,0.766880
